In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
import zipfile
import json

# 1. 전처리된 CSV 로드
df_orig = pd.read_csv('/content/drive/MyDrive/2026 4-1 캡스톤/DATA/TR_p2_original.csv')

print(f"전체 건수: {len(df_orig)}")
print(f"컬럼: {df_orig.columns.tolist()}")

# 2. 카테고리 확인
print("\n=== 카테고리별 분포 ===")
print(df_orig['newsCategory'].value_counts())

# 3. 스포츠 있는지 확인
print("\n=== 정상 기사 카테고리별 건수 ===")
print(df_orig[df_orig['clickbaitClass'] == 0]['newsCategory'].value_counts())

전체 건수: 295275
컬럼: ['newsID', 'newsCategory', 'newsSubcategory', 'newsTitle', 'newsContent', 'partNum', 'processType', 'processLevel', 'sentenceCount', 'clickbaitClass', 'processSentencenum', 'insertPositionRatio', 'insertedSent', 'prevSent', 'nextSent']

=== 카테고리별 분포 ===
newsCategory
사회       59132
세계       45738
경제       41082
IT&과학    40684
정치       40441
연예       36307
생활&문화    31891
Name: count, dtype: int64

=== 정상 기사 카테고리별 건수 ===
newsCategory
사회       30848
세계       24357
IT&과학    20784
경제       20414
정치       20383
연예       18759
생활&문화    14383
Name: count, dtype: int64


# 1단계. 도메인 간, 쉬운 버전
- 연예 기사 + 정치 문장 삽입(가장 이질적인 조합)

- 낚시성 데이터 1500 건 만드는 방
1. 연예 정상 기사 1500개 뽑기
2. 정치 기사에서 문장을 랜덤으로 하나 뽑기
3. 연예기사 앞쪽에 그 정치 문장 십입
4. 라벨 = 1(낚시성


- 정상 데이터 1500 건
1. 연예 기사 원본 그대로
2. 라벨 = 9

In [ ]:
import pandas as pd
import random

random.seed(42)

# 정상 기사 카테고리별 분리
df_normal = df_orig[df_orig['clickbaitClass'] == 0].copy()

# 연예 기사 (삽입 대상)
df_yeonje = df_normal[df_normal['newsCategory'] == '연예'].reset_index(drop=True)

# 정치 기사 (삽입할 문장 소스)
df_politics = df_normal[df_normal['newsCategory'] == '정치'].reset_index(drop=True)

print(f"연예 정상 기사: {len(df_yeonje)}")
print(f"정치 정상 기사: {len(df_politics)}")

# 정치 기사에서 문장 풀 만들기
politics_sentences = []
for _, row in df_politics.iterrows():
    sents = [s.strip() for s in row['newsContent'].split('\n') if len(s.strip()) > 20]
    politics_sentences.extend(sents)

print(f"정치 문장 풀: {len(politics_sentences)}개")

# 낚시성 데이터 생성 (1500건)
fake_rows = []
sample_yeonje = df_yeonje.sample(1500, random_state=42).reset_index(drop=True)

for _, row in sample_yeonje.iterrows():
    sentences = [s.strip() for s in row['newsContent'].split('\n') if len(s.strip()) > 20]
    if len(sentences) < 3:
        continue

    # 랜덤 삽입 위치 (앞쪽 25% 이내 — EDA 결과 반영)
    insert_idx = random.randint(0, max(0, len(sentences)//4))
    insert_sent = random.choice(politics_sentences)

    sentences.insert(insert_idx, insert_sent)
    new_content = '\n'.join(sentences)

    fake_rows.append({
        'newsID': row['newsID'] + '_fake',
        'newsCategory': row['newsCategory'],
        'newsTitle': row['newsTitle'],
        'newsContent': new_content,
        'clickbaitClass': 1
    })

df_fake_new = pd.DataFrame(fake_rows)

# 정상 데이터 (1500건)
df_normal_new = df_yeonje.sample(1500, random_state=99)[['newsID', 'newsCategory', 'newsTitle', 'newsContent']].copy()
df_normal_new['clickbaitClass'] = 0

# 합치기
df_test_new = pd.concat([df_fake_new, df_normal_new]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n=== 생성된 데이터 ===")
print(f"전체: {len(df_test_new)}")
print(df_test_new['clickbaitClass'].value_counts())

# 저장
df_test_new.to_csv('/content/drive/MyDrive/2026 4-1 캡스톤/DATA/test_yeonje_politics.csv', index=False)
print("\n저장 완료")

연예 정상 기사: 18759
정치 정상 기사: 20383
정치 문장 풀: 252386개

=== 생성된 데이터 ===
전체: 3000
clickbaitClass
0    1500
1    1500
Name: count, dtype: int64

저장 완료


In [ ]:
# 낚시성 샘플 1개
fake_sample = df_test_new[df_test_new['clickbaitClass'] == 1].iloc[0]
print(f"제목: {fake_sample['newsTitle']}\n")
sentences = [s for s in fake_sample['newsContent'].split('\n') if s.strip()]
for i, s in enumerate(sentences, 1):
    print(f"{i}. {s}")

제목: 서세원 홍콩 출국 \"여성 동행 보도 기사 명백한 허위…매우 화나\"

1. 방송인 서세원이 의문의 여인과 출국한 것으로 보도되면서 서세원이 이에 해명했다.
2. 지난해 서세원과 서정희의 이혼 소송 당시 서정희 측은 서세원의 외도 때문에 문제가 불거졌다고 주장했으나 서세원은 여자 문제는 절대 아니라고 주장했다.
3. 외교부 당국자는 27일 \"현재 이란 전국에서 코로나19 확진자가 발생하고 있고, 중국 다음으로 사망자가 많아 교민 안전을 살펴보고 있다\"며 \"주이란 한국대사관에서 교민의 귀국 의사 등을 파악 중\"이라고 말했다.
4. 서정희는 지난해 JTBC ‘연예특종’과의 인터뷰에서 “여자(내연녀)가 제 딸 아이 또래”라며 “(서세원이) 수도 없이 여자와 문자를 하고 지우고 계속 여자와 연락을 주고 받았다”고 주장했다.
5. 또 서정희는 “내연녀에게 매일 협박 문자를 받고 있다. 딸에게는 남편이 음성 메시지로 하루에 30통 이상씩 협박하고 있다”며 불안한 심경을 밝혔다.
6. 이에 서세원의 변호인은 이혼 이유에 대해 “서세원씨가 한 교회의 목사임에도 불구하고 서정희씨가 다른 교회를 다녀 불화가 시작된 것이지 여자 문제는 절대 아니다”고 말한 바 있다.
7. 이후 오늘 오전 한 매체에 따르면 서세원은 오전 6시 10분쯤 인천공항 출국장에 나타났으며 오전 8시 10분발 홍콩행 KE603 대한한공 비행기에 탑승하는 그는 카운터에서 한 여인과 만났다고 보도했다.
8. 이 매체는 서세원에게 다가가 \"(여성과) 같이 여행을 떠나는 것이냐\"고 질문하자 \"아니다. 내가 어디로 여행을 가겠나. 공항에 누구 좀 만나러 왔다\"고 말했다고 전했다.
9. 이후 기자가 서세원과 이야기를 주고받은 여성에게 다가가 \"서세원씨와 얘기를 나누는 것을 봤는데 두 분이 무슨 사이냐\"는 물음에 \"상관없는 사람\"이라고 대답했다고 밝혔다.
10. 한편 서세원은 8일 일간스포츠와의 전화통화에서 \"이제 막 홍콩에 도착했는데, 매우 화가난다. 현재 흥분한 상태\"라며 \"오전에 

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/2026 4-1 캡스톤/'))

['3 30 회의.gdoc', '2026 캡스톤디자인 3_30 발표자료.pdf', '1.gdoc', 'DATA', '멀티학회학술발표대회-논문양식.hwp', '4 1 회의록.gdoc', '4 2 회의록.gdoc', '4 5 회의록.gdoc', '4 6 회의록.gdoc', 'capstone_presentation.pptx', '[심가연, 이수현] 4_13 캡스톤 2차 발표.pdf', '4 4 회의록.gdoc', '모델 비교 정리.gdoc', '5 2 회의록.gdoc', '5 7 회의록 - 대본.gdoc', '공모전.gdoc', '5 20.gdoc', 'FalseNews(NEW).ipynb', '5 20 진행상황.gdoc', 'stop.ipynb', '5 23 진행 상황.gdoc', '6 1 교수님의 제안.gdoc']


In [ ]:
print(os.listdir('/content/drive/MyDrive/2026 4-1 캡스톤/DATA/'))

['TR_preprocessed.csv', 'VL_preprocessed.csv', 'kobert_p1_save.pt', 'Pp_Pipeline.ipynb', 'Validation_labeled', 'Training_labeled', 'Training_Original', 'Validation_Original', 'TR_p2_original.csv', 'VL_p2_original.csv', 'train_pairs.csv', 'val_pairs.csv', 'train_pairs_v2.csv', 'val_pairs_v2.csv', 'train_pairs_v3.csv', 'val_pairs_v3.csv', 'test_yeonje_politics.csv']


## KPF-BERT학습 돌리기

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import numpy as np

# 1. 데이터 로드
df = pd.read_csv('/content/drive/MyDrive/2026 4-1 캡스톤/DATA/test_yeonje_politics.csv')
print(f"전체: {len(df)}")
print(df['clickbaitClass'].value_counts())

# 2. train/val 분리 (80:20)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['clickbaitClass'])
print(f"\ntrain: {len(train_df)} / val: {len(val_df)}")

# 3. 데이터셋 클래스
class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['newsTitle']) + ' [SEP] ' + str(row['newsContent'])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(row['clickbaitClass'], dtype=torch.long)
        }

# 4. 모델 로드
MODEL_NAME = 'jinmang2/kpfbert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n사용 디바이스: {device}")
model.to(device)

# 5. 데이터로더
train_dataset = NewsDataset(train_df, tokenizer)
val_dataset = NewsDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# 6. 학습
optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # 7. 평가
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs.logits, dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(labels.cpu().numpy())

    f1 = f1_score(trues, preds)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | F1: {f1:.4f}")

print("\n=== 최종 결과 ===")
print(classification_report(trues, preds, target_names=['정상', '낚시성']))

전체: 3000
clickbaitClass
0    1500
1    1500
Name: count, dtype: int64

train: 2400 / val: 600


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/276k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: jinmang2/kpfbert
Key                      | Status  | 
-------------------------+---------+-
classifier.bias          | MISSING | 
bert.pooler.dense.weight | MISSING | 
bert.pooler.dense.bias   | MISSING | 
classifier.weight        | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



사용 디바이스: cpu


In [ ]:
# 다시
from google.colab import drive
drive.mount('/')


In [ ]:
import pandas as pd
import random
random.seed(42)

# 데이터 로드
df_orig = pd.read_csv('/kaggle/input/datasets/tlarkdus/part2-original/')
df_normal = df_orig[df_orig['clickbaitClass'] == 0].copy()

# 연예 기사
df_yeonje = df_normal[df_normal['newsCategory'] == '연예'].reset_index(drop=True)
# 정치 기사 문장 풀
df_politics = df_normal[df_normal['newsCategory'] == '정치'].reset_index(drop=True)

politics_sentences = []
for _, row in df_politics.iterrows():
    sents = [s.strip() for s in row['newsContent'].split('\n') if len(s.strip()) > 20]
    politics_sentences.extend(sents)

print(f"연예 정상 기사: {len(df_yeonje)}")
print(f"정치 문장 풀: {len(politics_sentences)}")

# 낚시성 1500건 — 라벨 0 (기존 실험과 동일)
fake_rows = []
sample_yeonje = df_yeonje.sample(1500, random_state=42).reset_index(drop=True)

for _, row in sample_yeonje.iterrows():
    sentences = [s.strip() for s in row['newsContent'].split('\n') if len(s.strip()) > 20]
    if len(sentences) < 3:
        continue
    insert_idx = random.randint(0, max(0, len(sentences)//4))
    insert_sent = random.choice(politics_sentences)
    sentences.insert(insert_idx, insert_sent)
    fake_rows.append({
        'newsID': row['newsID'] + '_fake',
        'newsCategory': row['newsCategory'],
        'newsTitle': row['newsTitle'],
        'newsContent': '\n'.join(sentences),
        'clickbaitClass': 0  # 낚시성 = 0
    })

df_fake_new = pd.DataFrame(fake_rows)

# 정상 1500건 — 라벨 1 (기존 실험과 동일)
df_normal_new = df_yeonje.sample(1500, random_state=99)[['newsID', 'newsCategory', 'newsTitle', 'newsContent']].copy()
df_normal_new['clickbaitClass'] = 1  # 정상 = 1

# 합치기
df_testA = pd.concat([df_fake_new, df_normal_new]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n=== 테스트셋 A ===")
print(f"전체: {len(df_testA)}")
print(df_testA['clickbaitClass'].value_counts())

# 저장
df_testA.to_csv('/kaggle/working/testA_yeonje_politics.csv', index=False)
print("\n저장 완료: testA_yeonje_politics.csv")